# Whole-brain MVPA: 4-way decoding + pattern stability

Two complementary multivariate analyses, both directly addressing the hypothesis that BDD has impaired lowSF-lowC representations.

## Option B: 4-way searchlight decoding
Train a linear SVM to distinguish all 4 conditions simultaneously within each searchlight sphere. For each subject, save:
- **Overall accuracy map** — global representational quality
- **Per-condition TPR maps** — how well each individual condition is decoded against the others

Group comparison on the lowSF-lowC TPR map directly tests "is lowSF-lowC less well-represented in BDD?"

## Option C: Within-condition pattern stability
For each condition, compute the pairwise spatial correlation between blocks within a searchlight sphere. High mean correlation = stable, reliable representation; low = noisy.

Group comparison on the lowSF-lowC stability map directly tests "are BDD's lowSF-lowC patterns noisier?"

Both produce per-subject NIfTI maps you can group-compare with second-level t-tests.

**Run time**: 4-way decoding is the slower of the two (~30–60 min per subject). Stability is much faster (~10 min per subject). Full batch over 59 subjects: probably 2 days for 4-way, half a day for stability — plan accordingly.

## 1. Install / import

In [ ]:
%pip install nilearn nibabel scikit-learn pandas joblib scipy --quiet

In [2]:
from pathlib import Path
import numpy as np
import nibabel as nib
import pandas as pd
from datetime import datetime

from scipy.special import gamma as gamma_fn
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import NearestNeighbors
from joblib import Parallel, delayed

## 2. Configuration

In [3]:
# --- Paths ---------------------------------------------------------------
FMRIPREP_DIR = Path("/Volumes/drive/AVP-BDD/derivatives")
BEHAV_DIR    = Path("/Volumes/drive/AVP-BDD/behavior")
OUT_DIR_4WAY = Path("/Volumes/drive/AVP-BDD/derivatives/mvpa_4way")
OUT_DIR_STAB = Path("/Volumes/drive/AVP-BDD/derivatives/mvpa_stability")
OUT_DIR_4WAY.mkdir(parents=True, exist_ok=True)
OUT_DIR_STAB.mkdir(parents=True, exist_ok=True)

# --- Acquisition ---------------------------------------------------------
TR_SECONDS = 1.0
TASK       = "SFlow"
SPACE      = "MNI152NLin2009cAsym"

CONDITION_MAP = {
    "Condition 1": "lowSF-highC",
    "Condition 2": "highSF-highC",
    "Condition 3": "lowSF-lowC",
    "Condition 4": "highSF-lowC",
}
CONDITIONS = ["lowSF-highC", "highSF-highC", "lowSF-lowC", "highSF-lowC"]

# --- Searchlight parameters ----------------------------------------------
RADIUS_MM = 8.0      # ~8 mm radius -> ~33 voxels at 3 mm
N_JOBS    = -1       # -1 = all CPU cores

# --- Subjects ------------------------------------------------------------
GROUP_1_SUBS = [s for s in range(102, 133) if s != 114]    # BDD, n=30
GROUP_2_SUBS = [s for s in range(201, 231) if s != 203]    # HC,  n=29
RUNS = [1, 2, 3]

print(f"BDD: {len(GROUP_1_SUBS)} subjects")
print(f"HC:  {len(GROUP_2_SUBS)} subjects")
print(f"Output (4-way):    {OUT_DIR_4WAY}")
print(f"Output (stability): {OUT_DIR_STAB}")

BDD: 30 subjects
HC:  29 subjects
Output (4-way):    /Volumes/drive/AVP-BDD/derivatives/mvpa_4way
Output (stability): /Volumes/drive/AVP-BDD/derivatives/mvpa_stability


## 3. Shared helpers (HRF, design matrix, beta extraction)

In [4]:
def double_gamma_hrf(t, a1=6.0, b1=1.0, a2=16.0, b2=1.0, c=1/6.0):
    """SPM-style canonical double-gamma HRF."""
    h = ((t**a1 * np.exp(-t/b1)) / (b1**(a1+1) * gamma_fn(a1+1))
         - c * (t**a2 * np.exp(-t/b2)) / (b2**(a2+1) * gamma_fn(a2+1)))
    h[t < 0] = 0
    return h


def build_lsa_design(behav_csv, n_tp, tr):
    """Least-squares-all design: one HRF-convolved column per BLOCK."""
    df = pd.read_csv(behav_csv)
    blocks = (df.groupby("Block")
                .agg(onset=("Stimulus Onset (s)",  "min"),
                     offset=("Stimulus Offset (s)", "max"),
                     condition=("Condition",        "first"))
                .reset_index().sort_values("onset"))

    fine_dt  = 0.05
    duration = n_tp * tr
    hrf      = double_gamma_hrf(np.arange(0, 32, fine_dt))
    n_fine   = int(duration / fine_dt)

    X = np.zeros((n_tp, len(blocks)))
    labels = []
    for col, (_, b) in enumerate(blocks.iterrows()):
        stick = np.zeros(n_fine)
        stick[int(b.onset/fine_dt):int(b.offset/fine_dt)] = 1.0
        conv = np.convolve(stick, hrf)[:n_fine]
        tr_idx = ((np.arange(n_tp) + 0.5) * tr / fine_dt).astype(int)
        tr_idx = np.clip(tr_idx, 0, n_fine - 1)
        X[:, col] = conv[tr_idx]
        labels.append(CONDITION_MAP[b.condition])
    return X, labels


def find_bold(sub_id, run):
    sub_root = FMRIPREP_DIR / sub_id
    glob = f"**/{sub_id}*_task-{TASK}_run-{run:02d}_space-{SPACE}_desc-preproc_bold.nii.gz"
    hits = list(sub_root.glob(glob))
    return hits[0] if hits else None


def find_brain_mask(sub_id, run):
    sub_root = FMRIPREP_DIR / sub_id
    glob = f"**/{sub_id}*_task-{TASK}_run-{run:02d}_space-{SPACE}_desc-brain_mask.nii.gz"
    hits = list(sub_root.glob(glob))
    return hits[0] if hits else None


def build_subject_betas(sub_num):
    """Voxelwise OLS with LSA design. Returns 4D Nifti1Image, labels, run IDs."""
    sub_id = f"sub-{sub_num}"
    raw_id = str(sub_num)

    all_betas, all_labels, all_runs, ref_img = [], [], [], None
    for run in RUNS:
        bold  = find_bold(sub_id, run)
        behav = BEHAV_DIR / raw_id / "low-level" / f"Subject_{raw_id}_Run_{run}_RT.csv"
        if bold is None or not behav.exists():
            print(f"  {sub_id} run-{run}: missing files, skipping"); continue

        img  = nib.load(str(bold))
        if ref_img is None:
            ref_img = img
        data = img.get_fdata(dtype=np.float32)
        n_tp = data.shape[3]

        # Z-transform per voxel time course
        m = data.mean(axis=3, keepdims=True); s = data.std(axis=3, keepdims=True)
        s[s == 0] = 1
        data = (data - m) / s

        X, labels = build_lsa_design(behav, n_tp, TR_SECONDS)

        XtX_inv_Xt = np.linalg.pinv(X.T @ X) @ X.T            # (n_blocks, T)
        flat_ts    = data.reshape(-1, n_tp).T                  # (T, n_voxels)
        flat_betas = XtX_inv_Xt @ flat_ts                      # (n_blocks, n_voxels)
        betas      = flat_betas.T.reshape(*data.shape[:3], -1) # (X, Y, Z, n_blocks)

        all_betas.append(betas)
        all_labels.extend(labels)
        all_runs.extend([run] * len(labels))

    if not all_betas:
        return None, None, None

    full = np.concatenate(all_betas, axis=3)
    return (nib.Nifti1Image(full, ref_img.affine, ref_img.header),
            np.array(all_labels), np.array(all_runs))


def get_searchlight_neighbors(mask_data, radius_mm, voxel_size):
    """Return list of in-mask voxel coords + per-voxel neighbor indices."""
    mask_coords = np.argwhere(mask_data)
    radius_voxels = radius_mm / voxel_size
    nn = NearestNeighbors(radius=radius_voxels).fit(mask_coords)
    neighbor_idx = nn.radius_neighbors(mask_coords, return_distance=False)
    return mask_coords, neighbor_idx

## 4. Option B — 4-way decoding searchlight

For each searchlight sphere, train a 4-class SVM with leave-one-run-out CV. Save:
- **overall accuracy** (chance = 0.25)
- **per-condition TPR** (true positive rate for each condition; chance = 0.25)

TPR for a given condition is the diagonal of that condition's row in the confusion matrix — the fraction of that condition's blocks correctly classified as itself.

In [ ]:
def _decode_one_sphere_4way(sphere_betas, labels, runs):
    """Run leave-one-run-out 4-way SVM decoding on one sphere's pattern.
    Returns (overall_acc, per_cond_tpr_dict)."""
    cv = LeaveOneGroupOut()
    cm_total = np.zeros((4, 4), dtype=int)
    accs = []
    for tr_idx, te_idx in cv.split(sphere_betas, labels, groups=runs):
        clf = make_pipeline(StandardScaler(), LinearSVC(C=1.0, max_iter=5000))
        clf.fit(sphere_betas[tr_idx], labels[tr_idx])
        preds = clf.predict(sphere_betas[te_idx])
        accs.append((preds == labels[te_idx]).mean())
        cm_total += confusion_matrix(labels[te_idx], preds, labels=CONDITIONS)
    overall = float(np.mean(accs))
    row_sums = cm_total.sum(axis=1)
    tpr = {c: (cm_total[i, i] / row_sums[i]) if row_sums[i] > 0 else 0.0
           for i, c in enumerate(CONDITIONS)}
    return overall, tpr


def run_4way_subject(sub_num, skip_existing=True):
    sub_id = f"sub-{sub_num}"

    overall_path  = OUT_DIR_4WAY / f"{sub_id}_4way_overall.nii.gz"
    per_cond_paths = {c: OUT_DIR_4WAY / f"{sub_id}_4way_{c}_TPR.nii.gz" for c in CONDITIONS}

    if skip_existing and overall_path.exists() and all(p.exists() for p in per_cond_paths.values()):
        print(f"=== {sub_id}: 4-way maps exist, skipping")
        return

    print(f"\n=== {sub_id}: 4-way decoding ===")
    t0 = datetime.now()

    betas_img, labels, runs = build_subject_betas(sub_num)
    if betas_img is None:
        print(f"  no data"); return

    mask_path = find_brain_mask(sub_id, 1)
    if mask_path is None:
        print(f"  no brain mask, skipping"); return

    mask_data = nib.load(str(mask_path)).get_fdata() > 0.5
    voxel_size = abs(betas_img.affine[0, 0])
    mask_coords, neighbor_idx = get_searchlight_neighbors(mask_data, RADIUS_MM, voxel_size)
    print(f"  {len(mask_coords)} voxels in mask, betas shape {betas_img.shape}")

    flat_betas = betas_img.get_fdata()[mask_data].T   # (n_blocks, n_mask_voxels)

    def _process(vox_idx):
        nb = neighbor_idx[vox_idx]
        if len(nb) < 5:
            return 0.0, {c: 0.0 for c in CONDITIONS}
        return _decode_one_sphere_4way(flat_betas[:, nb], labels, runs)

    print(f"  running searchlight in parallel...")
    results = Parallel(n_jobs=N_JOBS, verbose=1)(
        delayed(_process)(i) for i in range(len(mask_coords))
    )

    overall_arr = np.zeros(mask_data.shape, dtype=np.float32)
    per_cond_arr = {c: np.zeros(mask_data.shape, dtype=np.float32) for c in CONDITIONS}
    for (x, y, z), (ovr, tpr) in zip(mask_coords, results):
        overall_arr[x, y, z] = ovr
        for c in CONDITIONS:
            per_cond_arr[c][x, y, z] = tpr[c]

    nib.save(nib.Nifti1Image(overall_arr, betas_img.affine, betas_img.header), overall_path)
    print(f"  saved {overall_path.name}  (max acc = {overall_arr.max():.3f})")
    for c in CONDITIONS:
        nib.save(nib.Nifti1Image(per_cond_arr[c], betas_img.affine, betas_img.header),
                 per_cond_paths[c])
        print(f"  saved {per_cond_paths[c].name}  (max TPR = {per_cond_arr[c].max():.3f})")
    print(f"  total time: {datetime.now() - t0}")

## 5. Option C — within-condition pattern stability

For each condition, compute the **mean pairwise spatial correlation** between blocks of that condition within the searchlight sphere. 

- High correlation (closer to 1) = the brain produces a consistent pattern each time it sees that condition — stable representation.
- Low correlation (closer to 0) = patterns are noisy and inconsistent across blocks.

Output per subject: 4 NIfTI maps, one per condition. Group-compare to test whether BDD has noisier representations.

In [7]:
from joblib import Parallel, delayed
import numpy as np

# -----------------------------------------------------------------------------
# Vectorized stability over a chunk of voxels
# -----------------------------------------------------------------------------
def _stability_chunk(neighbor_idx_chunk, flat_betas, labels):
    """
    Process a chunk of voxels at once.
    Returns: dict mapping condition -> array of stability values, one per voxel in chunk.
    """
    n_vox = len(neighbor_idx_chunk)
    out = {c: np.zeros(n_vox, dtype=np.float32) for c in CONDITIONS}

    # Pre-compute demeaned + normalized beta patterns ONCE for all voxels
    # We'll subset to spheres on the fly.
    for cond_idx, cond in enumerate(CONDITIONS):
        sel = labels == cond
        if sel.sum() < 2:
            continue
        # Patterns for this condition: (n_cond_blocks, n_mask_voxels)
        cond_patterns = flat_betas[sel]
        n_blocks_cond = cond_patterns.shape[0]

        for vi, nb in enumerate(neighbor_idx_chunk):
            if len(nb) < 5:
                continue
            P = cond_patterns[:, nb]                       # (n_blocks_cond, n_sphere)
            P = P - P.mean(axis=1, keepdims=True)           # demean each pattern
            norms = np.linalg.norm(P, axis=1, keepdims=True)
            norms[norms == 0] = 1
            P = P / norms
            sim = P @ P.T                                   # (n_blocks_cond, n_blocks_cond)
            iu = np.triu_indices(n_blocks_cond, k=1)
            out[cond][vi] = sim[iu].mean()
    return out

def find_gm_mask(sub_id):
    """Find fMRIPrep's gray matter probability map."""
    sub_root = FMRIPREP_DIR / sub_id
    # Try various GM mask names fMRIPrep produces
    for pattern in [
        f"**/{sub_id}*space-{SPACE}_label-GM_probseg.nii.gz",
        f"**/{sub_id}*space-{SPACE}_desc-aparcaseg_dseg.nii.gz",
    ]:
        hits = list(sub_root.glob(pattern))
        if hits:
            return hits[0]
    return None

def run_stability_subject(sub_num, skip_existing=True, chunk_size=500):
    sub_id = f"sub-{sub_num}"
    out_paths = {c: OUT_DIR_STAB / f"{sub_id}_stability_{c}.nii.gz" for c in CONDITIONS}
    if skip_existing and all(p.exists() for p in out_paths.values()):
        print(f"=== {sub_id}: stability maps exist, skipping")
        return

    print(f"\n=== {sub_id}: pattern stability ===")
    t0 = datetime.now()

    betas_img, labels, runs = build_subject_betas(sub_num)
    if betas_img is None:
        print(f"  no data"); return

    mask_path = find_brain_mask(sub_id, 1)
    if mask_path is None:
        print(f"  no brain mask, skipping"); return

    mask_data = nib.load(str(mask_path)).get_fdata() > 0.5

    mask_data = nib.load(str(mask_path)).get_fdata() > 0.5

    # Optional: intersect with gray matter mask
    gm_path = find_gm_mask(sub_id)
    if gm_path is not None:
        gm_data = nib.load(str(gm_path)).get_fdata()
        if "probseg" in str(gm_path):
            mask_data &= (gm_data > 0.3)
        else:  # aparcaseg
            # Rough: gray matter labels are the cortical and subcortical ones
            # In Freesurfer aparcaseg: cortex ~ 1000-3000, subcortical ~ 10-60
            gm_only = ((gm_data >= 1000) & (gm_data < 3000)) | ((gm_data >= 10) & (gm_data <= 60))
            mask_data &= gm_only
        print(f"  intersected with GM: {mask_data.sum()} voxels remain")
        
    voxel_size = abs(betas_img.affine[0, 0])
    mask_coords, neighbor_idx = get_searchlight_neighbors(mask_data, RADIUS_MM, voxel_size)
    print(f"  {len(mask_coords)} voxels in mask")

    flat_betas = betas_img.get_fdata()[mask_data].T   # (n_blocks, n_mask_voxels)

    # Split voxels into chunks
    n_vox = len(mask_coords)
    chunks = [list(range(i, min(i + chunk_size, n_vox))) for i in range(0, n_vox, chunk_size)]
    print(f"  {len(chunks)} chunks of ~{chunk_size} voxels each")

    # Worker: takes a chunk of voxel indices, returns dict of stability arrays
    def _process_chunk(chunk):
        nb_chunk = [neighbor_idx[i] for i in chunk]
        return chunk, _stability_chunk(nb_chunk, flat_betas, labels)

    print(f"  running searchlight in parallel...")
    results = Parallel(n_jobs=N_JOBS, verbose=1, batch_size=1)(
        delayed(_process_chunk)(c) for c in chunks
    )

    arrays = {c: np.zeros(mask_data.shape, dtype=np.float32) for c in CONDITIONS}
    for chunk, chunk_out in results:
        for cond in CONDITIONS:
            for vi, vox_idx in enumerate(chunk):
                x, y, z = mask_coords[vox_idx]
                arrays[cond][x, y, z] = chunk_out[cond][vi]

    for c in CONDITIONS:
        nib.save(nib.Nifti1Image(arrays[c], betas_img.affine, betas_img.header), out_paths[c])
        print(f"  saved {out_paths[c].name}  (max = {arrays[c].max():.3f})")
    print(f"  total time: {datetime.now() - t0}")

## 6. Smoke test on sub-102

Run both analyses on one subject first. Verify the output looks reasonable before kicking off the full batch.

Reasonable values to expect:
- 4-way overall accuracy: chance = 0.25, informative spheres up to 0.5–0.8
- 4-way per-condition TPR: chance = 0.25, informative spheres up to 0.5–0.9
- Stability (correlation): near 0 in non-responsive areas, 0.2–0.7 in cortex

In [ ]:
run_4way_subject(102)

In [9]:
run_stability_subject(103)


=== sub-103: pattern stability ===


ValueError: operands could not be broadcast together with shapes (53,65,56) (193,229,193) (53,65,56) 

## 7. Full batch — all 59 subjects

Run them in two separate cells so you can launch one and let it finish before starting the other (or launch them on different machines / sessions). Stability is faster, so I'd start with that to get group-level results sooner.

In [ ]:
# Stability batch (faster)
t0 = datetime.now()
for sub_num in GROUP_1_SUBS + GROUP_2_SUBS:
    try:
        run_stability_subject(sub_num)
    except Exception as e:
        print(f"  ERROR sub-{sub_num}: {type(e).__name__}: {e}")
print(f"\n=== Stability batch done in {datetime.now() - t0} ===")

In [ ]:
# 4-way decoding batch (slower)
t0 = datetime.now()
for sub_num in GROUP_1_SUBS + GROUP_2_SUBS:
    try:
        run_4way_subject(sub_num)
    except Exception as e:
        print(f"  ERROR sub-{sub_num}: {type(e).__name__}: {e}")
print(f"\n=== 4-way batch done in {datetime.now() - t0} ===")

## 8. Completeness check

In [ ]:
rows = []
for grp, subs in [("BDD", GROUP_1_SUBS), ("HC", GROUP_2_SUBS)]:
    for s in subs:
        sub_id = f"sub-{s}"
        row = {"sub": s, "group": grp}
        # 4-way maps
        row["4way_overall"] = (OUT_DIR_4WAY / f"{sub_id}_4way_overall.nii.gz").exists()
        for c in CONDITIONS:
            row[f"4way_{c}_TPR"] = (OUT_DIR_4WAY / f"{sub_id}_4way_{c}_TPR.nii.gz").exists()
        # Stability maps
        for c in CONDITIONS:
            row[f"stab_{c}"] = (OUT_DIR_STAB / f"{sub_id}_stability_{c}.nii.gz").exists()
        rows.append(row)
summary = pd.DataFrame(rows)
complete_cols = [c for c in summary.columns if c not in ("sub", "group")]
summary["complete"] = summary[complete_cols].all(axis=1)
print(f"BDD complete: {summary[(summary.group=='BDD') & summary.complete].shape[0]}/{len(GROUP_1_SUBS)}")
print(f"HC  complete: {summary[(summary.group=='HC')  & summary.complete].shape[0]}/{len(GROUP_2_SUBS)}")
incomplete = summary[~summary.complete]
if len(incomplete):
    print("\nIncomplete:")
    print(incomplete[["sub", "group"] + [c for c in complete_cols if not incomplete[c].all()]].to_string(index=False))

## 9. Group comparison — generic helper

Same second-level approach as the original searchlight: per-subject map + group design matrix → voxelwise z-statistic. Use `direction='HC>BDD'` to test the hypothesis-consistent direction (controls show more / better, BDD shows less).

In [ ]:
from nilearn.glm.second_level import SecondLevelModel
from nilearn.glm import threshold_stats_img
from nilearn import plotting

def group_compare(map_template, out_label, direction="HC>BDD", out_dir=OUT_DIR_4WAY):
    """
    map_template: format string with {sub}, e.g. 'sub-{sub}_4way_lowSF-lowC_TPR.nii.gz'
                  (path is relative to out_dir)
    out_label   : suffix for the saved group-level map
    direction   : 'HC>BDD' (positive = HC higher) or 'BDD>HC'
    """
    maps, design = [], []
    for s in GROUP_1_SUBS + GROUP_2_SUBS:
        p = out_dir / map_template.format(sub=s)
        if not p.exists():
            continue
        maps.append(str(p))
        design.append({"BDD": int(s < 200), "HC": int(s >= 200)})
    design_df = pd.DataFrame(design)
    print(f"  {len(maps)} maps, BDD={design_df.BDD.sum()}, HC={design_df.HC.sum()}")

    contrast = [-1, 1] if direction == "HC>BDD" else [1, -1]
    slm  = SecondLevelModel().fit(maps, design_matrix=design_df)
    zmap = slm.compute_contrast(contrast, output_type="z_score")

    out_path = out_dir / f"group_{out_label}_{direction.replace('>','minus')}.nii.gz"
    nib.save(zmap, out_path)
    print(f"  saved {out_path.name}")
    return zmap, out_path

### 4-way decoding group comparisons

In [ ]:
# Per-condition TPR: where are individual conditions decoded better in HC than BDD?
print("=== 4-way per-condition TPR: HC > BDD ===")
for cond in CONDITIONS:
    print(f"\n--- {cond} ---")
    zmap, _ = group_compare(
        map_template = f"sub-{{sub}}_4way_{cond}_TPR.nii.gz",
        out_label    = f"4way_{cond}_TPR",
        direction    = "HC>BDD",
        out_dir      = OUT_DIR_4WAY,
    )

In [ ]:
# Overall 4-way accuracy: where is the overall representation poorer in BDD?
print("=== 4-way overall accuracy: HC > BDD ===")
zmap_overall, _ = group_compare(
    map_template = "sub-{sub}_4way_overall.nii.gz",
    out_label    = "4way_overall",
    direction    = "HC>BDD",
    out_dir      = OUT_DIR_4WAY,
)

### Stability group comparisons

In [ ]:
# Per-condition stability: where are individual conditions represented less stably in BDD?
print("=== Pattern stability: HC > BDD ===")
for cond in CONDITIONS:
    print(f"\n--- {cond} ---")
    zmap, _ = group_compare(
        map_template = f"sub-{{sub}}_stability_{cond}.nii.gz",
        out_label    = f"stability_{cond}",
        direction    = "HC>BDD",
        out_dir      = OUT_DIR_STAB,
    )

## 10. Visualize the hypothesis-relevant results

In [ ]:
# 4-way: HC > BDD on lowSF-lowC TPR (the hypothesis-relevant test)
zmap_path = OUT_DIR_4WAY / "group_4way_lowSF-lowC_TPR_HCminusBDD.nii.gz"
if zmap_path.exists():
    plotting.plot_stat_map(
        str(zmap_path), threshold=3.1,
        title="4-way TPR  HC > BDD  on lowSF-lowC  (z, p<0.001 unc)",
        display_mode="ortho", cut_coords=(0, -70, 0), cmap="hot",
    )

In [ ]:
# Stability: HC > BDD on lowSF-lowC patterns (also hypothesis-relevant)
zmap_path = OUT_DIR_STAB / "group_stability_lowSF-lowC_HCminusBDD.nii.gz"
if zmap_path.exists():
    plotting.plot_stat_map(
        str(zmap_path), threshold=3.1,
        title="Stability  HC > BDD  on lowSF-lowC  (z, p<0.001 unc)",
        display_mode="ortho", cut_coords=(0, -70, 0), cmap="hot",
    )

## 11. Cluster-FWE correction (permutation test)

Strict cluster-FWE correction via sign-flipping permutation. Produces a -log10(p) map where significant clusters have values above -log10(0.05) = 1.3.

In [ ]:
from nilearn.glm.second_level import non_parametric_inference

def permutation_test(map_template, out_label, direction="HC>BDD",
                     n_perm=1000, voxel_p=0.001, out_dir=OUT_DIR_4WAY):
    maps, design = [], []
    for s in GROUP_1_SUBS + GROUP_2_SUBS:
        p = out_dir / map_template.format(sub=s)
        if not p.exists():
            continue
        maps.append(str(p))
        design.append({"BDD": int(s < 200), "HC": int(s >= 200)})
    design_df = pd.DataFrame(design)
    contrast  = [-1, 1] if direction == "HC>BDD" else [1, -1]

    print(f"  {len(maps)} maps, {n_perm} permutations...")
    out = non_parametric_inference(
        maps,
        design_matrix         = design_df,
        second_level_contrast = contrast,
        n_perm                = n_perm,
        threshold             = voxel_p,
        n_jobs                = -1,
    )
    out_path = out_dir / f"group_{out_label}_{direction.replace('>','minus')}_clusterFWE.nii.gz"
    nib.save(out["logp_max_size"], out_path)
    print(f"  saved cluster-FWE-corrected -log10(p): {out_path.name}")
    return out

# Run for the two hypothesis-relevant maps (each ~10-30 min)
# Uncomment when ready:
# permutation_test("sub-{sub}_4way_lowSF-lowC_TPR.nii.gz", "4way_lowSF-lowC_TPR",
#                  out_dir=OUT_DIR_4WAY)
# permutation_test("sub-{sub}_stability_lowSF-lowC.nii.gz", "stability_lowSF-lowC",
#                  out_dir=OUT_DIR_STAB)